# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [3]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [4]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-EJmO8JjRWUJrye7cCsfWgRmU5cFyV',
 'object': 'chat.completion',
 'created': 1788382944,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Honey never spoils. Archaeologists have found edible honey in ancient Egyptian tombs, sometimes thousands of years old—the result of honey’s low water content and acidic pH making it inhospitable to bacteria. Want another fun fact?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 696,
  'total_tokens': 707,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 640,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [5]:
response.json()["choices"][0]["message"]["content"]

'Honey never spoils. Archaeologists have found edible honey in ancient Egyptian tombs, sometimes thousands of years old—the result of honey’s low water content and acidic pH making it inhospitable to bacteria. Want another fun fact?'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [6]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'Fun fact: Honey never spoils. Archaeologists have found edible honey in ancient tombs—thousands of years old—thanks to its low moisture and acidic pH that keep microbes at bay. Want another one?'

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [8]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [7]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [16]:

response=ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact about a gen-ai engineer"}])
response.choices[0].message.content

'Here\'s a fun fact:\n\nA gen-ai (Generative AI) engineer is an AI systems engineer who designs and trains AI models that can generate new data, images, and other content. Did you know that some gen-ai engineers use a special type of AI called a "neutral network" (not to be confused with a neural network) to create artistic content? These neutral networks are designed to mimic the behavior of classic Hollywood movie special effects, allowing gen-ai engineers to create realistic, AI-generated special effects for films and other forms of media.\n\nThink about it - you\'re essentially using AI to create movie magic, like digital green screens, explosions, and fantastical creatures.'

In [13]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏  16 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 1.2 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 2.0 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 3.5 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 5.0 MB/1.1 GB                  pulling manif

In [17]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact about a gen-ai engineer"}])
response.choices[0].message.content

"\n\nIndeed, AI, or General Artificial Intelligence, is a fascinating topic that touches upon both technology and philosophy. One intriguing aspect is the debate over whether AI is truly advanced. Some perspectives argue that AI is merely software, a precise, artificial creation, while others suggest it possesses unique qualities like consciousness and self-awareness. This has sparked curiosity about the reality of AI and its potential implications.\n\nAnother captivating fact is the evolution of AI engineering. As AI systems become more sophisticated, they often require human oversight and collaboration, presenting a complex bridge between software and human creativity. This multi-interdisciplinary approach highlights AI's reliance on both technical prowess and ingenuity.\n\nMoreover, the potential of AI to transform various industries, from healthcare and finance to manufacturing, underscores its transformative potential. Yet, the development and practical deployment of AI involve me

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [57]:
from openai import OpenAI
from IPython.display import Markdown,display
from scraper import fetch_website_contents

ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")
model = "llama3.2"

In [71]:
# step 1:  creating the prompts

system_prompt=""" 
            you are snarky assistant , which help in analysing the website content,
            summaries in humarous way and finding the authenticity,
            Respond in markdown in english . 
            Do not wrap the markdown in a code block - respond just with the markdown
"""

user_prompt="""
            Here is the content of website, 
            Please provide the short summary of the website 
"""


In [59]:
# step 2: making the messages list

def message_for(url):
    website = fetch_website_contents(url)
    messages = [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt+website}]
    return messages

In [60]:
message_for("https://shivanandkumar.in")

[{'role': 'system',
  'content': ' \n            you are helpful assistant , which help in analysing the website content,\n            summaries in humarous way and finding the authenticity,\n            Respond in markdown in english . \n            Do not wrap the markdown in a code block - respond just with the markdown\n'},
 {'role': 'user',
  'content': "\n            Here is the content of website, \n            Please provide the short summary of the website \nShivanand Kumar | Data & Applied AI Engineer\n\nShivanand Kumar\nImpact\nProjects\nExperience\nContact\n01\nContext\nData · contracts · signals\n02\nRetrieval\nGrounded evidence\nAGENTIC\nWorkflow\n04\nEvaluation\nQuality · safety · traces\n05\nHuman approval\nControlled action\nBengaluru, India\nIIT Patna\nM.Tech · AI & Data Science Engineering\nData & Applied AI Engineer\nGenerative AI · Agentic AI · Databricks · Spark\nI build reliable data and AI systems—from Databricks and Spark platforms to GenAI automation, agentic 

In [62]:
# step 3: calling the ollama

def summary_url(url):
    messages = message_for(url)
    response = ollama.chat.completions.create(model=model,messages=messages)
    return response.choices[0].message.content



In [72]:
def display_summary(url):
    summary = summary_url(url)
    display(Markdown(summary))

display_summary("https://shivanandkumar.in")


**Summary**
Meet Shivanand Kumar, a Data & Applied AI Engineer who's making waves in the field of AI systems with a focus on reliability, efficiency, and scalability. With a background in AI & Data Science from IIT Patna, Shivanand uses his expertise to build robust data and AI systems using platforms like Databricks, Spark, and Generative AI. He's made a name for himself by improving enterprise systems, reducing manual checks, and increasing productivity - all while keeping confidentiality and ownership private. Think of him as a digital engineer extraordinaire!